In [ ]:
import argparse
import logging
import inspect
import math
import os
from typing import Dict, Optional, Tuple

import diffusers
import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
import transformers
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import set_seed
from diffusers import AutoencoderKL, DDPMScheduler, DDIMScheduler
from diffusers.optimization import get_scheduler
from diffusers.utils.import_utils import is_xformers_available
from einops import rearrange
from omegaconf import OmegaConf
from tqdm import tqdm
from transformers import CLIPTextModel, CLIPTokenizer

from models.unet import UNet3DConditionModel
from pipelines.pipeline_tuneavideo import TuneAVideoPipeline
from tuneavideo.data.dataset import TuneMultiVideoDataset
from tuneavideo.util import save_videos_grid


# 设置 PyTorch CUDA 内存分配策略，以减少碎片
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:24"

logger = get_logger(__name__, log_level="INFO")


def setup_logging(accelerator):
    """配置日志记录器。

    Args:
        accelerator (Accelerator): The accelerator object.
    """
    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        level=logging.INFO,
    )
    logger.info(accelerator.state, main_process_only=False)
    if accelerator.is_local_main_process:
        transformers.utils.logging.set_verbosity_warning()
        diffusers.utils.logging.set_verbosity_info()
    else:
        transformers.utils.logging.set_verbosity_error()
        diffusers.utils.logging.set_verbosity_error()


def get_training_indices(num_blocks=7, num_classes=40):
    """
    根据预定义的标签顺序为训练数据生成索引。

    Args:
        num_blocks (int): 数据块的数量。
        num_classes (int): 类别数量。

    Returns:
        numpy.ndarray: 用于训练的视频样本索引数组。
    """
    # SEED-V 数据集中的视频类别顺序
    gt_label = np.array([
        [23, 22, 9, 6, 18, 14, 5, 36, 25, 19, 28, 35, 3, 16, 24, 40, 15, 27, 38, 33, 34, 4, 39, 17, 1, 26, 20, 29, 13, 32, 37, 2, 11, 12, 30, 31, 8, 21, 7, 10],
        [27, 33, 22, 28, 31, 12, 38, 4, 18, 17, 35, 39, 40, 5, 24, 32, 15, 13, 2, 16, 34, 25, 19, 30, 23, 3, 8, 29, 7, 20, 11, 14, 37, 6, 21, 1, 10, 36, 26, 9],
        [15, 36, 31, 1, 34, 3, 37, 12, 4, 5, 21, 24, 14, 16, 39, 20, 28, 29, 18, 32, 2, 27, 8, 19, 13, 10, 30, 40, 17, 26, 11, 9, 33, 25, 35, 7, 38, 22, 23, 6],
        [16, 28, 23, 1, 39, 10, 35, 14, 19, 27, 37, 31, 5, 18, 11, 25, 29, 13, 20, 24, 7, 34, 26, 4, 40, 12, 8, 22, 21, 30, 17, 2, 38, 9, 3, 36, 33, 6, 32, 15],
        [18, 29, 7, 35, 22, 19, 12, 36, 8, 15, 28, 1, 34, 23, 20, 13, 37, 9, 16, 30, 2, 33, 27, 21, 14, 38, 10, 17, 31, 3, 24, 39, 11, 32, 4, 25, 40, 5, 26, 6],
        [29, 16, 1, 22, 34, 39, 24, 10, 8, 35, 27, 31, 23, 17, 2, 15, 25, 40, 3, 36, 26, 6, 14, 37, 9, 12, 19, 30, 5, 28, 32, 4, 13, 18, 21, 20, 7, 11, 33, 38],
        [38, 34, 40, 10, 28, 7, 1, 37, 22, 9, 16, 5, 12, 36, 20, 30, 6, 15, 35, 2, 31, 26, 18, 24, 8, 3, 23, 19, 14, 13, 21, 4, 25, 11, 32, 17, 39, 29, 33, 27]
    ]) - 1  # 转换为0-based索引
    
    all_labels = np.vstack([gt.repeat(5) for gt in gt_label])
    
    # 假设使用第一个 block 的标签顺序来筛选所有视频
    # 在实际应用中，每个 block 可能需要不同的索引
    chosen_labels = np.arange(num_classes)
    return np.where(np.isin(all_labels[0], chosen_labels))[0]


def prepare_dataloader(train_data_config, tokenizer, selected_indices):
    """
    准备并配置训练数据集和数据加载器。

    Args:
        train_data_config (Dict): 包含数据路径和参数的配置字典。
        tokenizer (CLIPTokenizer): 用于编码文本提示的分词器。
        selected_indices (numpy.ndarray): 要用于训练的样本索引。

    Returns:
        torch.utils.data.DataLoader: 配置好的训练数据加载器。
    """
    train_dataset = TuneMultiVideoDataset(**train_data_config)
    
    # 根据索引动态加载视频文件和提示
    video_dir = train_data_config.video_path.rsplit('/', 1)[0]
    video_files = [os.path.join(video_dir, f"{i+1}.mp4") for i in selected_indices]
    
    all_prompts = []
    with open(train_data_config.prompt_path, 'r') as f:
        all_prompts = [line.strip() for line in f]
    
    selected_prompts = [all_prompts[i] for i in selected_indices]

    train_dataset.video_path = video_files
    train_dataset.prompt = selected_prompts
    
    logger.info(f"加载了 {len(video_files)} 个视频和 {len(selected_prompts)} 个提示进行训练。")

    # 预处理提示
    train_dataset.prompt_ids = tokenizer(
        list(train_dataset.prompt),
        max_length=tokenizer.model_max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).input_ids

    return torch.utils.data.DataLoader(
        train_dataset, batch_size=train_data_config.train_batch_size, shuffle=True
    )

def load_models(pretrained_model_path, trainable_modules):
    """
    加载所有必要的模型（VAE, Text Encoder, UNet）并设置可训练参数。

    Args:
        pretrained_model_path (str): 预训练 Stable Diffusion 模型的路径。
        trainable_modules (Tuple[str]): 需要解冻并训练的 UNet 模块名称后缀。

    Returns:
        Tuple: 包含 tokenizer, text_encoder, vae, unet, noise_scheduler 的元组。
    """
    noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_path, subfolder="scheduler")
    tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_path, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(pretrained_model_path, subfolder="text_encoder")
    vae = AutoencoderKL.from_pretrained(pretrained_model_path, subfolder="vae")
    unet = UNet3DConditionModel.from_pretrained_2d(pretrained_model_path, subfolder="unet")

    # 冻结 VAE 和 Text Encoder
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)

    # 仅解冻指定的 UNet 模块
    unet.requires_grad_(False)
    for name, module in unet.named_modules():
        if name.endswith(tuple(trainable_modules)):
            for params in module.parameters():
                params.requires_grad = True
    
    return tokenizer, text_encoder, vae, unet, noise_scheduler


def main(config):
    """
    主训练函数。

    Args:
        config (OmegaConf): 从 YAML 文件加载的配置对象。
    """
    accelerator = Accelerator(
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        mixed_precision=config.mixed_precision,
    )

    setup_logging(accelerator)

    if config.seed is not None:
        set_seed(config.seed)

    if accelerator.is_main_process:
        os.makedirs(config.output_dir, exist_ok=True)
        os.makedirs(os.path.join(config.output_dir, "samples"), exist_ok=True)
        OmegaConf.save(config, os.path.join(config.output_dir, 'config.yaml'))

    # 加载模型
    tokenizer, text_encoder, vae, unet, noise_scheduler = load_models(
        config.pretrained_model_path, config.trainable_modules
    )

    if config.enable_xformers_memory_efficient_attention:
        if is_xformers_available():
            unet.enable_xformers_memory_efficient_attention()
        else:
            raise ValueError("xformers is not available. Make sure it is installed correctly.")

    if config.gradient_checkpointing:
        unet.enable_gradient_checkpointing()

    # 创建优化器
    optimizer_cls = torch.optim.AdamW
    if config.use_8bit_adam:
        try:
            import bitsandbytes as bnb
            optimizer_cls = bnb.optim.AdamW8bit
        except ImportError:
            raise ImportError("Please install bitsandbytes to use 8-bit Adam (`pip install bitsandbytes`)")

    optimizer = optimizer_cls(
        filter(lambda p: p.requires_grad, unet.parameters()),
        lr=config.learning_rate,
        betas=(config.adam_beta1, config.adam_beta2),
        weight_decay=config.adam_weight_decay,
        eps=config.adam_epsilon,
    )

    # 准备数据
    selected_indices = get_training_indices()
    train_dataloader = prepare_dataloader(config.train_data, tokenizer, selected_indices)
    
    num_train_epochs = config.num_train_epochs
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / config.gradient_accumulation_steps)
    num_training_steps = num_train_epochs * num_update_steps_per_epoch

    # 创建学习率调度器
    lr_scheduler = get_scheduler(
        config.lr_scheduler,
        optimizer=optimizer,
        num_warmup_steps=config.lr_warmup_steps * config.gradient_accumulation_steps,
        num_training_steps=num_training_steps * accelerator.num_processes,
    )

    # 准备 Accelerator
    unet, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
        unet, optimizer, train_dataloader, lr_scheduler
    )

    weight_dtype = torch.float32
    if accelerator.mixed_precision == "fp16":
        weight_dtype = torch.float16
    elif accelerator.mixed_precision == "bf16":
        weight_dtype = torch.bfloat16

    text_encoder.to(accelerator.device, dtype=weight_dtype)
    vae.to(accelerator.device, dtype=weight_dtype)

    if accelerator.is_main_process:
        accelerator.init_trackers("text2video-fine-tune")

    # --- 开始训练 ---
    logger.info("***** Running training *****")
    logger.info(f"  Num examples = {len(train_dataloader.dataset)}")
    logger.info(f"  Num Epochs = {num_train_epochs}")
    logger.info(f"  Instantaneous batch size per device = {config.train_data.train_batch_size}")
    logger.info(f"  Gradient Accumulation steps = {config.gradient_accumulation_steps}")

    global_step = 0
    for epoch in range(num_train_epochs):
        unet.train()
        train_loss = 0.0

        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(unet):
                pixel_values = batch["pixel_values"].to(weight_dtype)
                video_length = pixel_values.shape[1]
                
                with torch.no_grad():
                    pixel_values_flat = rearrange(pixel_values, "b f c h w -> (b f) c h w")
                    latents = vae.encode(pixel_values_flat).latent_dist.sample()
                    latents = rearrange(latents, "(b f) c h w -> b c f h w", f=video_length)
                    latents = latents * vae.config.scaling_factor

                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=latents.device).long()

                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                with torch.no_grad():
                    encoder_hidden_states = text_encoder(batch["prompt_ids"])[0]

                if noise_scheduler.config.prediction_type == "epsilon":
                    target = noise
                elif noise_scheduler.config.prediction_type == "v_prediction":
                    target = noise_scheduler.get_velocity(latents, noise, timesteps)
                else:
                    raise ValueError(f"Unknown prediction type {noise_scheduler.config.prediction_type}")

                model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
                loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")

                avg_loss = accelerator.gather(loss.repeat(config.train_data.train_batch_size)).mean()
                train_loss += avg_loss.item() / config.gradient_accumulation_steps

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(filter(lambda p: p.requires_grad, unet.parameters()), config.max_grad_norm)
                
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            if accelerator.sync_gradients:
                global_step += 1
                accelerator.log({"train_loss": train_loss}, step=global_step)
                train_loss = 0.0

        # --- 验证和保存检查点 ---
        if accelerator.is_main_process and (epoch + 1) % config.validation_steps == 0:
            logger.info(f"Running validation for epoch {epoch + 1}...")
            unet_for_pipeline = accelerator.unwrap_model(unet)
            
            pipeline = TuneAVideoPipeline(
                vae=vae,
                text_encoder=text_encoder,
                tokenizer=tokenizer,
                unet=unet_for_pipeline,
                scheduler=DDIMScheduler.from_config(noise_scheduler.config),
            )
            pipeline.enable_vae_slicing()

            generator = torch.Generator(device=accelerator.device).manual_seed(config.seed)
            samples = []
            for prompt in config.validation_data.prompts:
                with torch.autocast("cuda"):
                    video = pipeline(prompt, generator=generator, **config.validation_data).videos
                samples.append(video)
                save_videos_grid(video, f"{config.output_dir}/samples/epoch-{epoch+1}-{prompt[:30].replace(' ', '_')}.gif")
            
            logger.info(f"Saved validation samples for epoch {epoch+1}")
            
            if (epoch + 1) % config.checkpointing_steps == 0:
                pipeline.save_pretrained(os.path.join(config.output_dir, f"checkpoint-{epoch+1}"))
                logger.info(f"Saved checkpoint to {config.output_dir}/checkpoint-{epoch+1}")


    # --- 训练结束 ---
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        unet = accelerator.unwrap_model(unet)
        pipeline = TuneAVideoPipeline.from_pretrained(
            config.pretrained_model_path,
            text_encoder=text_encoder,
            vae=vae,
            unet=unet,
        )
        pipeline.save_pretrained(config.output_dir)
        logger.info(f"Final model saved to {config.output_dir}")

    accelerator.end_training()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, default="./configs/all_40_video.yaml", help="Path to the training configuration file.")
    args = parser.parse_args()
    
    config = OmegaConf.load(args.config)
    main(config)